# IIEE Validation Experiment v3 — Kara Sea + Laptev Sea, OSI SAF

**Paper:** Implementation and Experimental Validation of a Spatial Error Decomposition Framework
**Regions:** Kara Sea (65–82°N, 55–100°E), Laptev Sea (70–82°N, 100–145°E)
**Data:** OSI SAF archive, daily SIC, 12.5 km EASE-Grid 2.0 (equal-area)
**Split:** Train 2019-01-01 to 2021-11-30, test 2022-01-01 to 2024-12-31 (1-month gap)
**Models:** Persistence baseline, Pixelwise LSTM, U-Net, ConvLSTM
**Metrics:** RMSE, MAE (traditional) vs IIEE, AEE, ME (spatial decomposition)

**Storage:** All data, models, and outputs saved to Google Drive.
Runtime estimate: ~4–6 hours on T4 GPU (first run with training); ~15 min on subsequent runs (cached).

In [ ]:
# ── 0. Environment detection + paths ─────────────────────────────────────────
from pathlib import Path

def detect_environment():
    try:
        import google.colab
        return 'colab'
    except ImportError:
        return 'local'

ENV = detect_environment()
print(f'Environment: {ENV}')

if ENV == 'colab':
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_ROOT = Path('/content/drive/MyDrive/iiee_experiment_v2')
    # Local scratch for temp files (faster than Drive FUSE)
    LOCAL_SCRATCH = Path('/content/scratch')
    LOCAL_SCRATCH.mkdir(exist_ok=True)
else:
    DRIVE_ROOT = Path.home() / 'research/data/iiee_experiment_v2'
    LOCAL_SCRATCH = DRIVE_ROOT

DATA_DIR   = DRIVE_ROOT / 'osisaf_archive'
OUT_DIR    = DRIVE_ROOT / 'outputs'
MODEL_DIR  = DRIVE_ROOT / 'models'

for d in [DATA_DIR, OUT_DIR, MODEL_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print(f'Drive root : {DRIVE_ROOT}')
print(f'Data dir   : {DATA_DIR}')
nc_count = len(list(DATA_DIR.glob("*.nc")))
print(f'Files on Drive: {nc_count} NetCDF, '
      f'{len(list(MODEL_DIR.glob("*.keras")))} models, '
      f'{len(list(OUT_DIR.glob("*")))} outputs')

In [ ]:
# ── 1. Dependencies ──────────────────────────────────────────────────────────
import subprocess, sys
deps = ['netCDF4', 'xarray', 'numpy', 'pandas', 'matplotlib', 'scikit-learn', 'tensorflow', 'scipy']
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q'] + deps)

In [ ]:
# ── 2. Download OSI SAF data via FTP to LOCAL disk ───────────────────────────
# Drive FUSE is completely unreliable for bulk reads/copies (crashes after ~20-1500 ops).
# Solution: download everything fresh from FTP to Colab's local SSD.
# FTP is fast (~5 MB/file, ~30 min for 2192 files) and 100% reliable.
# Only the small .npy cache (~170 MB) gets saved to Drive afterwards.

import ftplib
from datetime import date, timedelta

FTP_HOST = 'osisaf.met.no'

def osisaf_ftp_path(d: date) -> str:
    return f'/archive/ice/conc/{d.year}/{d.month:02d}/'

def osisaf_filename(d: date) -> str:
    return f'ice_conc_nh_ease-125_multi_{d.strftime("%Y%m%d")}1200.nc'

# LOCAL_NC_DIR: where NetCDF files are read from during processing
if ENV == 'colab':
    LOCAL_NC_DIR = Path('/content/osisaf_local')
    LOCAL_NC_DIR.mkdir(exist_ok=True)
else:
    LOCAL_NC_DIR = DATA_DIR  # already local

MIN_FILES = 2100

local_nc_count = len(list(LOCAL_NC_DIR.glob('*.nc')))
print(f'Local NetCDF files: {local_nc_count}')

if local_nc_count >= MIN_FILES:
    print(f'Sufficient data ({local_nc_count} files). Skipping FTP.')
else:
    print(f'Need ~{MIN_FILES}, have {local_nc_count}. Downloading via FTP to local disk...')
    print(f'Target: {LOCAL_NC_DIR}')
    try:
        with ftplib.FTP(FTP_HOST, timeout=60) as ftp:
            ftp.login()
            print('Connected to OSI SAF FTP (anonymous).')
            downloaded, skipped, errors = 0, 0, 0
            current = date(2019, 1, 1)
            end = date(2024, 12, 31)
            while current <= end:
                fname = osisaf_filename(current)
                local_path = LOCAL_NC_DIR / fname
                if local_path.exists() and local_path.stat().st_size > 10_000:
                    skipped += 1
                    current += timedelta(days=1)
                    continue
                if local_path.exists() and local_path.stat().st_size < 10_000:
                    local_path.unlink()
                try:
                    ftp.cwd(osisaf_ftp_path(current))
                    with open(local_path, 'wb') as f:
                        ftp.retrbinary(f'RETR {fname}', f.write)
                    downloaded += 1
                    if downloaded % 200 == 0:
                        print(f'  {downloaded} downloaded, {skipped} cached, {errors} missing...')
                except ftplib.error_perm:
                    errors += 1  # file doesn't exist on server (normal for some dates)
                except Exception as e:
                    errors += 1
                    if errors <= 5:
                        print(f'  Error {current}: {e}')
                    # Reconnect on connection errors
                    try:
                        ftp.quit()
                    except Exception:
                        pass
                    ftp = ftplib.FTP(FTP_HOST, timeout=60)
                    ftp.login()
                current += timedelta(days=1)
            print(f'FTP done. Downloaded: {downloaded}, cached: {skipped}, missing/errors: {errors}')
    except Exception as e:
        print(f'FTP connection failed: {e}')

# Step 2: Verify file integrity — delete corrupt files and re-download
# Files copied from Drive FUSE may be corrupt (valid size but bad content).
# NetCDF4/HDF5 files start with bytes \x89HDF.
print('\nVerifying file integrity...')
corrupt = []
for f in sorted(LOCAL_NC_DIR.glob('*.nc')):
    with open(f, 'rb') as fh:
        header = fh.read(4)
    if header[:3] != b'CDF' and header != b'\x89HDF':
        corrupt.append(f)
if corrupt:
    print(f'Found {len(corrupt)} corrupt files — deleting and re-downloading...')
    for f in corrupt:
        f.unlink()
    # Re-download corrupt files from FTP
    try:
        with ftplib.FTP(FTP_HOST, timeout=60) as ftp:
            ftp.login()
            redownloaded, ftp_missing = 0, 0
            for f in corrupt:
                # Extract date from filename: ice_conc_nh_ease-125_multi_YYYYMMDD1200.nc
                datestr = f.name.split('_')[-1][:8]  # YYYYMMDD
                d = date(int(datestr[:4]), int(datestr[4:6]), int(datestr[6:8]))
                try:
                    ftp.cwd(osisaf_ftp_path(d))
                    local_path = LOCAL_NC_DIR / f.name
                    with open(local_path, 'wb') as fh:
                        ftp.retrbinary(f'RETR {f.name}', fh.write)
                    redownloaded += 1
                except ftplib.error_perm:
                    ftp_missing += 1  # file genuinely missing from server
                except Exception as e:
                    ftp_missing += 1
                    if ftp_missing <= 3:
                        print(f'  Re-download error {d}: {e}')
                    try:
                        ftp.quit()
                    except Exception:
                        pass
                    ftp = ftplib.FTP(FTP_HOST, timeout=60)
                    ftp.login()
            print(f'  Re-downloaded: {redownloaded}, unavailable on server: {ftp_missing}')
    except Exception as e:
        print(f'  FTP reconnect failed: {e}')
else:
    print('All files valid.')

local_nc_count = len(list(LOCAL_NC_DIR.glob('*.nc')))
print(f'Final local NetCDF count: {local_nc_count}')

In [ ]:
# ── 3. Load and crop to study regions ────────────────────────────────────────
# EASE-Grid 2.0 is equal-area by design: cell area is exactly 156.25 km²
# (12.5 km × 12.5 km) everywhere. Ref: Brodzik et al. 2012.
# After first run, loads from Drive/.npy cache in ~10s.
# On Colab: Drive FUSE is unreliable for 2000+ sequential reads.
#   Strategy: try cache first, rebuild only if needed, never delete Drive files.

import xarray as xr
import numpy as np
from datetime import date, timedelta
import pickle, time

REGIONS = {
    'Kara Sea':   {'lat': (65, 82), 'lon': (55, 100)},
    'Laptev Sea': {'lat': (70, 82), 'lon': (100, 145)},
}

def find_cache(region_name: str):
    """Find .npy cache — checks both v3 name (kara_sea) and v2 name (kara)."""
    safe_name = region_name.lower().replace(' ', '_')
    # v3 naming: sic_kara_sea_125_2019_2024.npy
    candidates_sic = [
        DRIVE_ROOT / f'sic_{safe_name}_125_2019_2024.npy',
    ]
    candidates_dates = [
        DRIVE_ROOT / f'dates_{safe_name}_125_2019_2024.pkl',
    ]
    # v2 naming: sic_kara_125_2019_2024.npy (no "sea" suffix)
    short_name = safe_name.split('_')[0]  # "kara" from "kara_sea"
    if short_name != safe_name:
        candidates_sic.append(DRIVE_ROOT / f'sic_{short_name}_125_2019_2024.npy')
        candidates_dates.append(DRIVE_ROOT / f'dates_{short_name}_125_2019_2024.pkl')

    for sc, dc in zip(candidates_sic, candidates_dates):
        try:
            if sc.exists() and dc.exists():
                return sc, dc
        except OSError:
            continue
    return None, None

def load_region(region_name: str, lat_range: tuple, lon_range: tuple):
    """Load and crop SIC array for a region, with Drive caching."""
    safe_name = region_name.lower().replace(' ', '_')

    # 1. Try loading from existing cache (reject if too incomplete)
    MIN_CACHED_DAYS = 2000  # reject caches with fewer days (incomplete builds)
    sic_cache, dates_cache = find_cache(region_name)
    if sic_cache is not None:
        print(f'  Loading {region_name} from cache: {sic_cache.name}')
        try:
            sic = np.load(str(sic_cache))
            with open(dates_cache, 'rb') as f:
                dates = pickle.load(f)
            if len(dates) < MIN_CACHED_DAYS:
                print(f'  Cache has only {len(dates)} days (need {MIN_CACHED_DAYS}), rebuilding...')
            else:
                print(f'  {region_name}: {sic.shape} ({len(dates)} days)')
                return sic, dates
        except OSError as e:
            print(f'  Cache read failed ({e}), rebuilding from NetCDF...')

    # 2. Build from NetCDF files (always read from LOCAL disk, never Drive FUSE)
    print(f'  Building {region_name} from NetCDF (reading from {LOCAL_NC_DIR})...')
    nc_files = sorted(LOCAL_NC_DIR.glob('*.nc'))
    if not nc_files:
        raise RuntimeError(f'No NetCDF files found in {LOCAL_NC_DIR}')

    # Read grid coords from first file
    sample = xr.open_dataset(str(nc_files[0]), engine='netcdf4')
    lat = sample['lat'].values
    lon = sample['lon'].values
    sample.close()

    mask = ((lat >= lat_range[0]) & (lat <= lat_range[1]) &
            (lon >= lon_range[0]) & (lon <= lon_range[1]))
    rows_idx, cols_idx = np.where(mask)
    R0, R1 = rows_idx.min(), rows_idx.max() + 1
    C0, C1 = cols_idx.min(), cols_idx.max() + 1
    print(f'  Subgrid: {R1-R0} x {C1-C0} pixels')

    dates, frames = [], []
    current = date(2019, 1, 1)
    end_date = date(2024, 12, 31)
    errors = 0
    consecutive_errors = 0

    while current <= end_date:
        fname = osisaf_filename(current)
        path = LOCAL_NC_DIR / fname
        try:
            if not path.exists():
                current += timedelta(days=1)
                continue
            ds = xr.open_dataset(str(path), engine='netcdf4')
            sic_raw = ds['ice_conc'].values.squeeze()
            sic_raw = np.where(sic_raw > 100, np.nan, sic_raw / 100.0)
            ds.close()
            crop = np.nan_to_num(sic_raw[R0:R1, C0:C1], nan=0.0)
            frames.append(crop.astype(np.float32))
            dates.append(current)
            consecutive_errors = 0
            if len(frames) % 500 == 0:
                print(f'  ... {len(frames)} days loaded')
        except Exception as e:
            errors += 1
            consecutive_errors += 1
            if consecutive_errors <= 3:
                print(f'  Error {current}: {e}')
            elif consecutive_errors == 4:
                print(f'  (suppressing further errors...)')
        current += timedelta(days=1)

    if not frames:
        raise RuntimeError(f'No data loaded for {region_name}!')

    sic_arr = np.stack(frames, axis=0)
    print(f'  Loaded {len(dates)} days -> {sic_arr.shape} ({errors} errors)')

    # Save cache — on Colab, save to local scratch first, then copy to Drive
    out_sic   = DRIVE_ROOT / f'sic_{safe_name}_125_2019_2024.npy'
    out_dates = DRIVE_ROOT / f'dates_{safe_name}_125_2019_2024.pkl'
    if ENV == 'colab':
        tmp_sic   = LOCAL_SCRATCH / out_sic.name
        tmp_dates = LOCAL_SCRATCH / out_dates.name
        np.save(str(tmp_sic), sic_arr)
        with open(tmp_dates, 'wb') as f:
            pickle.dump(dates, f)
        # Copy to Drive (single large file — FUSE handles this OK)
        import shutil
        try:
            shutil.copy2(str(tmp_sic), str(out_sic))
            shutil.copy2(str(tmp_dates), str(out_dates))
            print(f'  Cache saved to Drive: {out_sic.name} ({sic_arr.nbytes / 1e6:.0f} MB)')
        except OSError as e:
            print(f'  Drive save failed ({e}), cache is at {tmp_sic}')
    else:
        np.save(str(out_sic), sic_arr)
        with open(out_dates, 'wb') as f:
            pickle.dump(dates, f)
        print(f'  Cache saved: {out_sic.name} ({sic_arr.nbytes / 1e6:.0f} MB)')
    return sic_arr, dates

# Load all regions
region_data = {}
for rname, rbox in REGIONS.items():
    print(f'\n=== {rname} ===')
    sic, dates = load_region(rname, rbox['lat'], rbox['lon'])
    region_data[rname] = {'SIC': sic, 'dates': dates}

# Verify completeness
for rname, rd in region_data.items():
    year_counts = {}
    for d in rd['dates']:
        year_counts[d.year] = year_counts.get(d.year, 0) + 1
    expected = {2019: 365, 2020: 366, 2021: 365, 2022: 365, 2023: 365, 2024: 366}
    missing = {y: expected[y] - year_counts.get(y, 0) for y in expected}
    total_missing = sum(missing.values())
    print(f'\n{rname}: {len(rd["dates"])} days, missing {total_missing}')
    if total_missing > 0:
        print(f'  Per year: {missing}')
    # Relaxed check: on Colab FUSE may lose some days, but we need most
    if total_missing > 100:
        print(f'  WARNING: {total_missing} missing days — results may be unreliable')
    assert total_missing < 200, f'Too many missing days in {rname}: {missing}'

# Use Kara Sea as primary for model development
SIC   = region_data['Kara Sea']['SIC']
dates = region_data['Kara Sea']['dates']
print(f'\nPrimary region: Kara Sea {SIC.shape}')
print(f'Date range: {dates[0]} to {dates[-1]}')

In [ ]:
# ── 4. IIEE decomposition functions ──────────────────────────────────────────
import numpy as np

THRESHOLD = 0.15        # 15% SIC binarization threshold (WMO standard)
CELL_AREA_KM2 = 156.25  # 12.5 km × 12.5 km — exact for EASE-Grid 2.0 (equal-area)

# Minimum meaningful IIEE: ignore trivially small edge-pixel shifts.
# 50 cells × 156.25 km² ≈ 7,800 km² — roughly a 90 km × 90 km region.
MIN_IIEE_KM2 = 50 * CELL_AREA_KM2

def binarize(sic: np.ndarray) -> np.ndarray:
    """B(x,y) = 1 if SIC >= 0.15, else 0."""
    return (sic >= THRESHOLD).astype(np.float32)

def compute_iiee(forecast: np.ndarray, reference: np.ndarray,
                 cell_area_km2: float = CELL_AREA_KM2) -> dict:
    """
    IIEE decomposition (Goessling et al. 2016).
    IIEE = AEE + ME
    AEE  = |Area(B_F) - Area(B_R)|           (extent bias)
    ME   = IIEE - AEE                         (misplacement error)
    """
    B_f = forecast.astype(np.float32)
    B_r = reference.astype(np.float32)
    area_f = B_f.sum() * cell_area_km2
    area_r = B_r.sum() * cell_area_km2
    aee  = abs(area_f - area_r)
    iiee = ((B_f != B_r).sum()) * cell_area_km2
    me   = iiee - aee
    ratio = me / iiee if iiee > 0 else 0.0
    return {'iiee': iiee, 'aee': aee, 'me': me, 'me_iiee_ratio': ratio}

def compute_rmse_mae(forecast_sic: np.ndarray, reference_sic: np.ndarray) -> dict:
    diff = forecast_sic - reference_sic
    return {'rmse': float(np.sqrt(np.mean(diff**2))), 'mae': float(np.mean(np.abs(diff)))}

print(f'IIEE functions ready. Cell area: {CELL_AREA_KM2} km² (EASE-Grid 2.0, equal-area)')
print(f'Min IIEE filter: {MIN_IIEE_KM2:.0f} km² ({int(MIN_IIEE_KM2/CELL_AREA_KM2)} cells)')

In [ ]:
# ── 5. Train/test split (1-month gap) ────────────────────────────────────────
from datetime import date
import pandas as pd

LEAD_DAYS = [1, 3, 5, 7, 10]
LOOKBACK  = 30

# Train: 2019-01-01 to 2021-11-30 | Gap: Dec 2021 | Test: 2022-01-01 to 2024-12-31
TRAIN_END  = date(2021, 11, 30)
TEST_START = date(2022, 1, 1)

train_idx = [i for i, d in enumerate(dates) if d <= TRAIN_END]
test_idx  = [i for i, d in enumerate(dates) if d >= TEST_START]

H, W = SIC.shape[1], SIC.shape[2]
SIC_train = SIC[:max(train_idx) + 1]

print(f'Train: {len(train_idx)} days (up to {TRAIN_END})')
print(f'Test:  {len(test_idx)} days (from {TEST_START})')
print(f'Gap:   Dec 2021 (31 days excluded)')
print(f'Grid:  {H}×{W} pixels')
print(f'Leads: {LEAD_DAYS}')

In [ ]:
# ── 6. Model A: Persistence baseline ────────────────────────────────────────
print('Running Persistence baseline...')

persistence_results = []
for lead in LEAD_DAYS:
    rows = []
    for i in test_idx:
        ref_i = i + lead
        if ref_i >= len(dates):
            continue
        trad    = compute_rmse_mae(SIC[i], SIC[ref_i])
        spatial = compute_iiee(binarize(SIC[i]), binarize(SIC[ref_i]))
        rows.append({'date': dates[i], 'lead': lead, 'model': 'Persistence', **trad, **spatial})
    persistence_results.extend(rows)
    df_r = pd.DataFrame(rows)
    print(f'  Lead {lead:2d}d  RMSE={df_r["rmse"].mean():.3f}  '
          f'IIEE={df_r["iiee"].mean():.0f} km²  ME/IIEE={df_r["me_iiee_ratio"].mean():.2f}')

df_persistence = pd.DataFrame(persistence_results)
print(f'Done. {len(df_persistence)} forecast-day pairs.')

In [ ]:
# ── 7. Model B: Pixelwise LSTM ──────────────────────────────────────────────
# Each pixel gets its own time-series prediction (shared weights).
# This produces genuinely different spatial patterns from Persistence.
import tensorflow as tf
import numpy as np
import gc

def build_pixel_lstm(lookback: int) -> tf.keras.Model:
    model = tf.keras.Sequential([
        tf.keras.layers.Input(shape=(lookback, 1)),
        tf.keras.layers.LSTM(32),
        tf.keras.layers.Dense(16, activation='relu'),
        tf.keras.layers.Dense(1, activation='sigmoid'),
    ])
    model.compile(optimizer='adam', loss='mse')
    return model

# Training: subsample pixels (every 3rd → ~11% of grid) for tractability
PIXEL_STEP = 3
pixel_coords = [(r, c) for r in range(0, H, PIXEL_STEP) for c in range(0, W, PIXEL_STEP)]
print(f'Training pixels: {len(pixel_coords)} / {H*W} ({100*len(pixel_coords)/(H*W):.0f}%)')

lstm_models = {}
for lead in LEAD_DAYS:
    model_path = MODEL_DIR / f'pxlstm_lead{lead:02d}.keras'

    if model_path.exists():
        print(f'Lead {lead}d — loading from Drive...')
        lstm_models[lead] = tf.keras.models.load_model(str(model_path))
        continue

    print(f'Lead {lead}d — building training data...')
    X_all, y_all = [], []
    # Sample every 5th timestep to keep dataset manageable
    for t in range(LOOKBACK, len(SIC_train) - lead, 5):
        for (r, c) in pixel_coords:
            X_all.append(SIC_train[t-LOOKBACK:t, r, c])
            y_all.append(SIC_train[t+lead, r, c])

    X_arr = np.array(X_all, dtype=np.float32)[..., np.newaxis]
    y_arr = np.array(y_all, dtype=np.float32)
    print(f'  Training samples: {len(X_arr):,}')

    model = build_pixel_lstm(LOOKBACK)
    model.fit(X_arr, y_arr, epochs=15, batch_size=256, verbose=0, validation_split=0.1)
    model.save(str(model_path))
    print(f'  Saved: {model_path.name}')
    lstm_models[lead] = model
    del X_arr, y_arr
    gc.collect()

print('All Pixelwise LSTM models ready.')

# Inference
print('\nRunning LSTM inference (pixelwise)...')
lstm_results = []
for lead in LEAD_DAYS:
    model = lstm_models[lead]
    rows = []
    for i in test_idx:
        ref_i = i + lead
        if ref_i >= len(SIC) or i < LOOKBACK:
            continue
        # Predict each pixel independently (vectorized over all pixels)
        seq = SIC[i-LOOKBACK:i]  # (LOOKBACK, H, W)
        # Reshape to (H*W, LOOKBACK, 1) for batch prediction
        X_batch = seq.transpose(1, 2, 0).reshape(H*W, LOOKBACK, 1)
        pred_flat = model.predict(X_batch, verbose=0, batch_size=4096).flatten()
        forecast = np.clip(pred_flat.reshape(H, W), 0, 1)

        trad    = compute_rmse_mae(forecast, SIC[ref_i])
        spatial = compute_iiee(binarize(forecast), binarize(SIC[ref_i]))
        rows.append({'date': dates[i], 'lead': lead, 'model': 'LSTM', **trad, **spatial})
    lstm_results.extend(rows)
    df_r = pd.DataFrame(rows)
    print(f'  Lead {lead:2d}d  RMSE={df_r["rmse"].mean():.3f}  '
          f'IIEE={df_r["iiee"].mean():.0f} km²  ME/IIEE={df_r["me_iiee_ratio"].mean():.2f}')

df_lstm = pd.DataFrame(lstm_results)
print(f'LSTM done. {len(df_lstm)} forecast-day pairs.')

In [ ]:
# ── 8. Model C: U-Net (spatial segmentation) ─────────────────────────────────
# Image-to-image: input = stack of recent SIC frames → output = SIC forecast.
# Learns spatial features → typically produces extent bias (high AEE, lower ME).
import tensorflow as tf
import numpy as np
import gc

# Pad grid to be divisible by 4 for pooling/upsampling
PAD_H = int(np.ceil(H / 4) * 4)
PAD_W = int(np.ceil(W / 4) * 4)
UNET_LOOKBACK = 10  # use last 10 days as input channels

def pad_frame(x, target_h, target_w):
    """Pad array to target size (bottom-right)."""
    if x.ndim == 2:
        return np.pad(x, ((0, target_h - x.shape[0]), (0, target_w - x.shape[1])))
    # (H, W, C)
    return np.pad(x, ((0, target_h - x.shape[0]), (0, target_w - x.shape[1]), (0, 0)))

def unpad_frame(x, orig_h, orig_w):
    return x[:orig_h, :orig_w]

def build_unet(h, w, channels):
    inputs = tf.keras.Input(shape=(h, w, channels))
    # Encoder
    c1 = tf.keras.layers.Conv2D(32, 3, padding='same', activation='relu')(inputs)
    c1 = tf.keras.layers.Conv2D(32, 3, padding='same', activation='relu')(c1)
    p1 = tf.keras.layers.MaxPooling2D(2)(c1)
    c2 = tf.keras.layers.Conv2D(64, 3, padding='same', activation='relu')(p1)
    c2 = tf.keras.layers.Conv2D(64, 3, padding='same', activation='relu')(c2)
    p2 = tf.keras.layers.MaxPooling2D(2)(c2)
    # Bottleneck
    c3 = tf.keras.layers.Conv2D(128, 3, padding='same', activation='relu')(p2)
    # Decoder
    u2 = tf.keras.layers.UpSampling2D(2)(c3)
    u2 = tf.keras.layers.Concatenate()([u2, c2])
    c4 = tf.keras.layers.Conv2D(64, 3, padding='same', activation='relu')(u2)
    u1 = tf.keras.layers.UpSampling2D(2)(c4)
    u1 = tf.keras.layers.Concatenate()([u1, c1])
    c5 = tf.keras.layers.Conv2D(32, 3, padding='same', activation='relu')(u1)
    outputs = tf.keras.layers.Conv2D(1, 1, activation='sigmoid')(c5)
    model = tf.keras.Model(inputs, outputs)
    model.compile(optimizer='adam', loss='mse')
    return model

print(f'U-Net grid: {H}x{W} -> padded {PAD_H}x{PAD_W}, {UNET_LOOKBACK} input channels')

unet_models = {}
for lead in LEAD_DAYS:
    model_path = MODEL_DIR / f'unet_lead{lead:02d}.keras'

    if model_path.exists():
        print(f'Lead {lead}d — loading from Drive...')
        unet_models[lead] = tf.keras.models.load_model(str(model_path))
        continue

    print(f'Lead {lead}d — building training data...')
    X_list, y_list = [], []
    for t in range(UNET_LOOKBACK, len(SIC_train) - lead, 3):  # stride 3 to limit size
        inp = SIC_train[t-UNET_LOOKBACK:t].transpose(1, 2, 0)  # (H, W, 10)
        out = SIC_train[t+lead]                                  # (H, W)
        X_list.append(pad_frame(inp, PAD_H, PAD_W))
        y_list.append(pad_frame(out, PAD_H, PAD_W)[..., np.newaxis])

    X_arr = np.array(X_list, dtype=np.float32)
    y_arr = np.array(y_list, dtype=np.float32)
    print(f'  Samples: {len(X_arr)}, shape: {X_arr.shape}')

    model = build_unet(PAD_H, PAD_W, UNET_LOOKBACK)
    model.fit(X_arr, y_arr, epochs=20, batch_size=4, verbose=0, validation_split=0.1)
    model.save(str(model_path))
    print(f'  Saved: {model_path.name}')
    unet_models[lead] = model
    del X_arr, y_arr
    gc.collect()

print('All U-Net models ready.')

# Inference
print('\nRunning U-Net inference...')
unet_results = []
for lead in LEAD_DAYS:
    model = unet_models[lead]
    rows = []
    for i in test_idx:
        ref_i = i + lead
        if ref_i >= len(SIC) or i < UNET_LOOKBACK:
            continue
        inp = SIC[i-UNET_LOOKBACK:i].transpose(1, 2, 0)
        inp_padded = pad_frame(inp, PAD_H, PAD_W)[np.newaxis]
        pred_padded = model.predict(inp_padded, verbose=0)[0, :, :, 0]
        forecast = np.clip(unpad_frame(pred_padded, H, W), 0, 1)

        trad    = compute_rmse_mae(forecast, SIC[ref_i])
        spatial = compute_iiee(binarize(forecast), binarize(SIC[ref_i]))
        rows.append({'date': dates[i], 'lead': lead, 'model': 'U-Net', **trad, **spatial})
    unet_results.extend(rows)
    df_r = pd.DataFrame(rows)
    print(f'  Lead {lead:2d}d  RMSE={df_r["rmse"].mean():.3f}  '
          f'IIEE={df_r["iiee"].mean():.0f} km²  ME/IIEE={df_r["me_iiee_ratio"].mean():.2f}')

df_unet = pd.DataFrame(unet_results)
print(f'U-Net done. {len(df_unet)} forecast-day pairs.')

In [ ]:
# ── 9. Model D: ConvLSTM (spatiotemporal) ────────────────────────────────────
# Processes sequence of SIC frames as spatiotemporal volume.
# Expected: best spatial coherence → lowest ME/IIEE ratio.
import tensorflow as tf
import numpy as np
import gc

CLSTM_LOOKBACK = 10  # reduced from 30 to fit GPU memory

def build_convlstm(h, w, lookback):
    inputs = tf.keras.Input(shape=(lookback, h, w, 1))
    x = tf.keras.layers.ConvLSTM2D(16, 3, padding='same', return_sequences=True,
                                    activation='relu')(inputs)
    x = tf.keras.layers.ConvLSTM2D(16, 3, padding='same', return_sequences=False,
                                    activation='relu')(x)
    x = tf.keras.layers.Conv2D(1, 1, activation='sigmoid')(x)
    model = tf.keras.Model(inputs, x)
    model.compile(optimizer='adam', loss='mse')
    return model

print(f'ConvLSTM grid: {PAD_H}x{PAD_W}, lookback={CLSTM_LOOKBACK}')

clstm_models = {}
for lead in LEAD_DAYS:
    model_path = MODEL_DIR / f'convlstm_lead{lead:02d}.keras'

    if model_path.exists():
        print(f'Lead {lead}d — loading from Drive...')
        clstm_models[lead] = tf.keras.models.load_model(str(model_path))
        continue

    print(f'Lead {lead}d — building training data...')
    X_list, y_list = [], []
    for t in range(CLSTM_LOOKBACK, len(SIC_train) - lead, 5):  # stride 5
        seq = SIC_train[t-CLSTM_LOOKBACK:t]  # (10, H, W)
        seq_padded = np.stack([pad_frame(f, PAD_H, PAD_W) for f in seq])
        X_list.append(seq_padded[..., np.newaxis])
        y_list.append(pad_frame(SIC_train[t+lead], PAD_H, PAD_W)[..., np.newaxis])

    X_arr = np.array(X_list, dtype=np.float32)
    y_arr = np.array(y_list, dtype=np.float32)
    print(f'  Samples: {len(X_arr)}, shape: {X_arr.shape}')

    model = build_convlstm(PAD_H, PAD_W, CLSTM_LOOKBACK)
    model.fit(X_arr, y_arr, epochs=15, batch_size=2, verbose=0, validation_split=0.1)
    model.save(str(model_path))
    print(f'  Saved: {model_path.name}')
    clstm_models[lead] = model
    del X_arr, y_arr
    gc.collect()

print('All ConvLSTM models ready.')

# Inference
print('\nRunning ConvLSTM inference...')
clstm_results = []
for lead in LEAD_DAYS:
    model = clstm_models[lead]
    rows = []
    for i in test_idx:
        ref_i = i + lead
        if ref_i >= len(SIC) or i < CLSTM_LOOKBACK:
            continue
        seq = SIC[i-CLSTM_LOOKBACK:i]
        seq_padded = np.stack([pad_frame(f, PAD_H, PAD_W) for f in seq])
        inp = seq_padded[np.newaxis, ..., np.newaxis]
        pred_padded = model.predict(inp, verbose=0)[0, :, :, 0]
        forecast = np.clip(unpad_frame(pred_padded, H, W), 0, 1)

        trad    = compute_rmse_mae(forecast, SIC[ref_i])
        spatial = compute_iiee(binarize(forecast), binarize(SIC[ref_i]))
        rows.append({'date': dates[i], 'lead': lead, 'model': 'ConvLSTM', **trad, **spatial})
    clstm_results.extend(rows)
    df_r = pd.DataFrame(rows)
    print(f'  Lead {lead:2d}d  RMSE={df_r["rmse"].mean():.3f}  '
          f'IIEE={df_r["iiee"].mean():.0f} km²  ME/IIEE={df_r["me_iiee_ratio"].mean():.2f}')

df_clstm = pd.DataFrame(clstm_results)
print(f'ConvLSTM done. {len(df_clstm)} forecast-day pairs.')

In [ ]:
# ── 10. Combine all results ──────────────────────────────────────────────────
df_all = pd.concat([df_persistence, df_lstm, df_unet, df_clstm], ignore_index=True)
print(f'Total forecast-day pairs: {len(df_all)}')
print(f'Models: {sorted(df_all["model"].unique())}')

# Verify models produce different IIEE values (critical check)
print('\n=== IIEE sanity check (must differ between models) ===')
for lead in LEAD_DAYS:
    sub = df_all[df_all['lead'] == lead]
    means = sub.groupby('model')['iiee'].mean()
    print(f'Lead {lead}d:')
    for m, v in means.items():
        print(f'  {m:12s} IIEE={v:.0f} km²')
    vals = means.values
    assert vals.max() - vals.min() > 100, f'Models produce near-identical IIEE at lead {lead}d!'

In [ ]:
# ── 11. Table 1: Metric disagreement cases ───────────────────────────────────
RMSE_THRESHOLD    = 0.10  # "acceptable" aggregate performance
FITNESS_THRESHOLD = 0.5   # ME/IIEE > 0.5: misplacement dominates → operationally unsafe
                          # Source: predecessor paper, grounded in Goessling et al. 2016:
                          # at 0.5, misplacement error equals extent error.

# Apply minimum area filter — ignore trivially small edge-pixel shifts
df_nontrivial = df_all[df_all['iiee'] >= MIN_IIEE_KM2].copy()
print(f'After area filter (>={MIN_IIEE_KM2:.0f} km²): '
      f'{len(df_nontrivial)}/{len(df_all)} pairs retained '
      f'({100*len(df_nontrivial)/len(df_all):.1f}%)')

disagreement = df_nontrivial[
    (df_nontrivial['rmse'] < RMSE_THRESHOLD) &
    (df_nontrivial['me_iiee_ratio'] > FITNESS_THRESHOLD)
].copy()

print(f'\nDisagreement cases (RMSE✓ but ME/IIEE✗): {len(disagreement)} '
      f'({100*len(disagreement)/len(df_nontrivial):.1f}% of non-trivial pairs)')

# Also report without filter for transparency
disagree_raw = df_all[
    (df_all['rmse'] < RMSE_THRESHOLD) &
    (df_all['me_iiee_ratio'] > FITNESS_THRESHOLD)
]
print(f'Without area filter: {len(disagree_raw)} ({100*len(disagree_raw)/len(df_all):.1f}%)')

print()
print(disagreement[['date','lead','model','rmse','mae','iiee','aee','me','me_iiee_ratio']]
      .sort_values('me_iiee_ratio', ascending=False).head(15).to_string(index=False))

disagreement.to_csv(OUT_DIR / 'table1_disagreement_cases.csv', index=False)

In [ ]:
# ── 12. Sensitivity analysis: threshold robustness ───────────────────────────
import matplotlib.pyplot as plt

# Sweep both thresholds to show the finding is robust
rmse_thresholds = [0.05, 0.08, 0.10, 0.12, 0.15]
me_thresholds   = np.arange(0.30, 0.75, 0.05)

df_nt = df_nontrivial  # use area-filtered data

print('Disagreement % (RMSE < threshold AND ME/IIEE > threshold):')
print(f'{"":>8s}', ''.join(f'ME>{t:.2f}  ' for t in me_thresholds))
sensitivity = {}
for rmse_th in rmse_thresholds:
    row = []
    line = f'RMSE<{rmse_th:.2f}'
    for me_th in me_thresholds:
        n = len(df_nt[(df_nt['rmse'] < rmse_th) & (df_nt['me_iiee_ratio'] > me_th)])
        pct = 100 * n / len(df_nt) if len(df_nt) > 0 else 0
        row.append(pct)
        line += f'  {pct:5.1f}%'
    sensitivity[rmse_th] = row
    print(line)

# Heatmap
fig, ax = plt.subplots(figsize=(10, 4))
data = np.array([sensitivity[r] for r in rmse_thresholds])
im = ax.imshow(data, cmap='YlOrRd', aspect='auto')
ax.set_xticks(range(len(me_thresholds)))
ax.set_xticklabels([f'{t:.2f}' for t in me_thresholds])
ax.set_yticks(range(len(rmse_thresholds)))
ax.set_yticklabels([f'{t:.2f}' for t in rmse_thresholds])
ax.set_xlabel('ME/IIEE threshold')
ax.set_ylabel('RMSE threshold')
ax.set_title('Metric Disagreement Rate (%) — Sensitivity to Threshold Choice')
for i in range(len(rmse_thresholds)):
    for j in range(len(me_thresholds)):
        ax.text(j, i, f'{data[i,j]:.1f}', ha='center', va='center', fontsize=8)
plt.colorbar(im, label='Disagreement %')
plt.tight_layout()
plt.savefig(str(OUT_DIR / 'figure4_sensitivity_heatmap.png'), dpi=150, bbox_inches='tight')
plt.show()
print(f'\nSaved: figure4_sensitivity_heatmap.png')

In [ ]:
# ── 13. Statistical rigor: bootstrap CIs and McNemar's test ──────────────────
from sklearn.utils import resample
from scipy.stats import chi2

def bootstrap_ci(series, stat_fn=np.mean, n_boot=1000, ci=95):
    """Bootstrap confidence interval."""
    boot = [stat_fn(resample(series, replace=True)) for _ in range(n_boot)]
    lo = np.percentile(boot, (100 - ci) / 2)
    hi = np.percentile(boot, 100 - (100 - ci) / 2)
    return lo, hi

def mcnemar_p(pass_a, pass_b):
    """McNemar's test for paired binary outcomes."""
    b = ((pass_a) & (~pass_b)).sum()
    c = ((~pass_a) & (pass_b)).sum()
    if b + c == 0:
        return 1.0
    stat = (abs(b - c) - 1)**2 / (b + c)
    return 1 - chi2.cdf(stat, df=1)

# Bootstrap CI on headline disagreement %
disagree_flags = ((df_nt['rmse'] < RMSE_THRESHOLD) &
                  (df_nt['me_iiee_ratio'] > FITNESS_THRESHOLD)).values
pct = 100 * disagree_flags.mean()
lo, hi = bootstrap_ci(disagree_flags, stat_fn=lambda x: 100*x.mean())
print(f'Disagreement rate: {pct:.1f}% (95% CI: [{lo:.1f}%, {hi:.1f}%])')

# Per-model CIs for key metrics
print('\n=== Table 2 with 95% CIs ===')
models = sorted(df_nt['model'].unique())
for model in models:
    for lead in LEAD_DAYS:
        sub = df_nt[(df_nt['model'] == model) & (df_nt['lead'] == lead)]
        if len(sub) == 0:
            continue
        rmse_m = sub['rmse'].mean()
        rmse_lo, rmse_hi = bootstrap_ci(sub['rmse'].values)
        iiee_m = sub['iiee'].mean()
        iiee_lo, iiee_hi = bootstrap_ci(sub['iiee'].values)
        me_r = sub['me_iiee_ratio'].mean()
        me_lo, me_hi = bootstrap_ci(sub['me_iiee_ratio'].values)
        print(f'{model:12s} lead={lead:2d}d  '
              f'RMSE={rmse_m:.3f}[{rmse_lo:.3f},{rmse_hi:.3f}]  '
              f'IIEE={iiee_m:.0f}[{iiee_lo:.0f},{iiee_hi:.0f}]  '
              f'ME/IIEE={me_r:.2f}[{me_lo:.2f},{me_hi:.2f}]')

# McNemar's test: do models differ in fitness pass/fail?
print('\n=== McNemar\'s test (fitness pass/fail, lead=7d) ===')
df7 = df_nt[df_nt['lead'] == 7].copy()
for i, m1 in enumerate(models):
    for m2 in models[i+1:]:
        # Align by date
        d1 = df7[df7['model'] == m1].set_index('date')['me_iiee_ratio']
        d2 = df7[df7['model'] == m2].set_index('date')['me_iiee_ratio']
        common = d1.index.intersection(d2.index)
        if len(common) < 10:
            continue
        pass1 = d1.loc[common] < FITNESS_THRESHOLD
        pass2 = d2.loc[common] < FITNESS_THRESHOLD
        p = mcnemar_p(pass1.values, pass2.values)
        agree = (pass1 == pass2).mean()
        print(f'  {m1} vs {m2}: p={p:.4f} (agree={100*agree:.0f}%)'
              f'{" ***" if p < 0.001 else " **" if p < 0.01 else " *" if p < 0.05 else ""}')

In [ ]:
# ── 14. Table 2: Aggregated metrics ──────────────────────────────────────────
summary = df_all.groupby(['model', 'lead']).agg(
    rmse_mean      = ('rmse',          'mean'),
    mae_mean       = ('mae',           'mean'),
    iiee_mean      = ('iiee',          'mean'),
    aee_mean       = ('aee',           'mean'),
    me_mean        = ('me',            'mean'),
    me_iiee_ratio  = ('me_iiee_ratio', 'mean'),
    fitness_pass   = ('me_iiee_ratio', lambda x: (x < FITNESS_THRESHOLD).mean()),
).reset_index()

print(summary.to_string(index=False))
summary.to_csv(OUT_DIR / 'table2_aggregated_metrics.csv', index=False)
print(f'\nSaved: table2_aggregated_metrics.csv')

In [ ]:
# ── 15. Figure 2: Metric degradation by lead time ────────────────────────────
import matplotlib.pyplot as plt

models = ['Persistence', 'LSTM', 'U-Net', 'ConvLSTM']
colors = {'Persistence': '#e6550d', 'LSTM': '#3182bd', 'U-Net': '#31a354', 'ConvLSTM': '#756bb1'}
markers = {'Persistence': 'o', 'LSTM': 's', 'U-Net': '^', 'ConvLSTM': 'D'}

fig, axes = plt.subplots(2, 2, figsize=(12, 8))
for (col, label), ax in zip([
    ('rmse_mean',     'RMSE'),
    ('mae_mean',      'MAE'),
    ('iiee_mean',     'IIEE (km²)'),
    ('me_iiee_ratio', 'ME/IIEE ratio'),
], axes.flat):
    for m in models:
        sub = summary[summary['model'] == m]
        ax.plot(sub['lead'], sub[col], marker=markers[m], label=m,
                color=colors[m], linewidth=2)
    ax.set_xlabel('Lead time (days)')
    ax.set_ylabel(label)
    ax.set_title(label)
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)
    if col == 'me_iiee_ratio':
        ax.axhline(0.5, color='red', ls='--', alpha=0.7, label='Fitness threshold')
        ax.legend(fontsize=8)

fig.suptitle('Figure 2: Metric Degradation by Lead Time — Kara Sea 2022–2024', fontsize=13)
plt.tight_layout()
plt.savefig(str(OUT_DIR / 'figure2_lead_time_degradation.png'), dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── 16. Figure 3: Seasonal breakdown (lead=7d) ───────────────────────────────
def season(d) -> str:
    m = d.month
    if m in (12, 1, 2):  return 'Winter'
    if m in (3, 4, 5):   return 'Spring (melt)'
    if m in (6, 7, 8):   return 'Summer'
    return 'Autumn (freeze-up)'

df_all['season'] = df_all['date'].apply(season)
df7 = df_all[df_all['lead'] == 7]
seasonal = df7.groupby(['model', 'season']).agg(
    rmse_mean     = ('rmse',          'mean'),
    me_iiee_ratio = ('me_iiee_ratio', 'mean'),
).reset_index()

season_order = ['Winter', 'Spring (melt)', 'Summer', 'Autumn (freeze-up)']
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for ax, (col, label) in zip(axes, [('rmse_mean', 'RMSE'), ('me_iiee_ratio', 'ME/IIEE')]):
    for m in models:
        sub = seasonal[seasonal['model'] == m].set_index('season').reindex(season_order)
        ax.plot(season_order, sub[col], marker=markers[m], label=m,
                color=colors[m], linewidth=2)
    ax.set_ylabel(label)
    ax.set_title(f'{label} by season (lead=7d)')
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)
    plt.setp(ax.xaxis.get_majorticklabels(), rotation=15, ha='right')
    if col == 'me_iiee_ratio':
        ax.axhline(0.5, color='red', ls='--', alpha=0.7)

fig.suptitle('Figure 3: Seasonal Validation — Kara Sea 2022–2024 (lead=7d)', fontsize=13)
plt.tight_layout()
plt.savefig(str(OUT_DIR / 'figure3_seasonal_breakdown.png'), dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── 17. Multi-region comparison (Laptev Sea) ─────────────────────────────────
# Run Persistence on Laptev Sea to show cross-region consistency.
# Full model training on Laptev is skipped for time — results section notes this.

if 'Laptev Sea' in region_data:
    SIC_L   = region_data['Laptev Sea']['SIC']
    dates_L = region_data['Laptev Sea']['dates']
    H_L, W_L = SIC_L.shape[1], SIC_L.shape[2]

    test_idx_L = [i for i, d in enumerate(dates_L) if d >= TEST_START]
    print(f'Laptev Sea: {SIC_L.shape}, {len(test_idx_L)} test days')

    laptev_results = []
    for lead in LEAD_DAYS:
        rows = []
        for i in test_idx_L:
            ref_i = i + lead
            if ref_i >= len(dates_L):
                continue
            trad    = compute_rmse_mae(SIC_L[i], SIC_L[ref_i])
            spatial = compute_iiee(binarize(SIC_L[i]), binarize(SIC_L[ref_i]))
            rows.append({'date': dates_L[i], 'lead': lead, 'model': 'Persistence',
                         'region': 'Laptev Sea', **trad, **spatial})
        laptev_results.extend(rows)

    df_laptev = pd.DataFrame(laptev_results)

    # Compare with Kara Sea Persistence
    df_kara_pers = df_persistence.copy()
    df_kara_pers['region'] = 'Kara Sea'
    df_laptev['region'] = 'Laptev Sea'
    df_cross = pd.concat([df_kara_pers, df_laptev], ignore_index=True)
    df_cross_nt = df_cross[df_cross['iiee'] >= MIN_IIEE_KM2]

    print('\n=== Cross-region Persistence comparison ===')
    cross_summary = df_cross_nt.groupby(['region', 'lead']).agg(
        rmse   = ('rmse', 'mean'),
        iiee   = ('iiee', 'mean'),
        me_r   = ('me_iiee_ratio', 'mean'),
    ).reset_index()
    print(cross_summary.to_string(index=False))

    # Disagreement rate per region
    for region in ['Kara Sea', 'Laptev Sea']:
        sub = df_cross_nt[df_cross_nt['region'] == region]
        dis = sub[(sub['rmse'] < RMSE_THRESHOLD) & (sub['me_iiee_ratio'] > FITNESS_THRESHOLD)]
        print(f'\n{region} disagreement: {len(dis)}/{len(sub)} '
              f'({100*len(dis)/len(sub):.1f}%)')
else:
    print('Laptev Sea data not loaded — skipping cross-region comparison.')

In [ ]:
# ── 18. Save all results + final summary ─────────────────────────────────────
df_all.to_csv(OUT_DIR / 'all_validation_results.csv', index=False)

print('=== OUTPUTS ON GOOGLE DRIVE ===')
for f in sorted(OUT_DIR.glob('*')):
    size_kb = f.stat().st_size // 1024
    print(f'  {f.name:50s} {size_kb:6d} KB')

print()
n_nt = len(df_nontrivial)
n_dis = len(disagreement)
lo, hi = bootstrap_ci(
    ((df_nt['rmse'] < RMSE_THRESHOLD) & (df_nt['me_iiee_ratio'] > FITNESS_THRESHOLD)).values,
    stat_fn=lambda x: 100*x.mean()
)
print(f'Key finding: in {100*n_dis/n_nt:.1f}% (95% CI: [{lo:.1f}%, {hi:.1f}%]) '
      f'of non-trivial cases ({n_dis}/{n_nt}),')
print(f'  RMSE < {RMSE_THRESHOLD} but ME/IIEE > {FITNESS_THRESHOLD}')
print()
print('Models tested: Persistence, Pixelwise LSTM, U-Net, ConvLSTM')
print('Regions: Kara Sea (primary) + Laptev Sea (Persistence cross-check)')
print('Statistical tests: bootstrap CIs, McNemar\'s test')
print()
print('→ Spatial decomposition reveals operationally unsafe forecasts')
print('  that aggregate metrics miss — across multiple architectures and regions.')